# Phase 3 - Hoang Grounded Output Research

Notebook này dùng **cùng pipeline với Streamlit** (`resolve_phase2_output` + `RUN_MODE`):

1. Load Phase 1 artifact.
2. Phase 2: artifact San → nếu không có thì **local FAISS retrieval ASRS** (giống app).
3. Phase 3: `Fast local` = local answer; `Full dense/Route LLM` = Route LLM nếu có API key.
4. Export artifact + **đánh giá grounding** (xem các section markdown cuối notebook).

### Cấu trúc phần đánh giá (cuối notebook)

| Section | Nội dung |
|---------|----------|
| **Hallucination-Risk Proxy** | Input / process / output; giải thích bảng overlap và số `0.2403` |
| **Đánh giá Phase 3** | Tiêu chí thống kê: citation, empty rate, avg risk |
| **Gold-set Evaluation** | Tiêu chí pass/fail trên gold labels |


In [1]:
from pathlib import Path
import re
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'aviation_rag').exists() and (ROOT.parent / 'aviation_rag').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from aviation_rag.config import Settings, ensure_artifact_dirs
from aviation_rag.io_utils import read_jsonl, write_jsonl
from aviation_rag.phase2_san_contract_adapter import MOCK_RETRIEVAL_LIBRARY, Phase2SanContractAdapter
from aviation_rag.phase2_san_faiss_retrieval import Phase2SanFaissRetrieval
from aviation_rag.phase3_hoang_grounded_qa import Phase3HoangGroundedQA
from aviation_rag.runtime import (
    RUN_MODES,
    force_local_for_run_mode,
    notebook_settings,
    resolve_phase2_output,
)
from aviation_rag.schemas import InputAgentOutput

# Đổi sang 'Full dense/Route LLM' để khớp Streamlit khi demo Route LLM
RUN_MODE = 'Fast local'
assert RUN_MODE in RUN_MODES

settings = notebook_settings(Settings(), run_mode=RUN_MODE)
ensure_artifact_dirs(settings)
phase2_retrieval = Phase2SanFaissRetrieval(settings)
phase2_adapter = Phase2SanContractAdapter(settings)

print('Run mode:', RUN_MODE)
print('force_local:', force_local_for_run_mode(RUN_MODE))
print('Embedding:', settings.phase2_embedding_model)
print('Index:', settings.phase2_index_dir)
print('Phase 1 artifact:', settings.phase1_output_path)
print('Phase 2 artifact:', settings.phase2_output_path)
print('Phase 3 artifact:', settings.phase3_output_path)
print('Retrieval available:', phase2_retrieval.available)
print('Mock library file: aviation_rag/phase2_san_contract_adapter.py')


Run mode: Fast local
force_local: True
Embedding: tfidf_svd_fallback
Index: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase2_index_fast
Phase 1 artifact: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase1_hoang_intent_routing_output.jsonl
Phase 2 artifact: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase2_san_retrieval_output.jsonl
Phase 3 artifact: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase3_hoang_grounded_answer_output.jsonl
Retrieval available: True
Mock library file: aviation_rag/phase2_san_contract_adapter.py


## Load Phase 1 Artifact

Phase 3 cần query gốc và intent từ Phase 1. Nếu artifact chưa tồn tại, chạy notebook Phase 1 trước.


In [2]:
phase1_rows = read_jsonl(settings.phase1_output_path)
if not phase1_rows:
    raise ValueError(f'No rows found in {settings.phase1_output_path}. Run phase 1 notebook first.')
phase1_rows[0]


{'query_id': 'q_incident_001',
 'query_raw': 'engine failure after takeoff with emergency return',
 'query_normalized': 'engine failure after takeoff with emergency return',
 'intent': 'Incident_Report',
 'intent_confidence': 0.5491248787888274,
 'intent_source': 'ml',
 'expanded_queries': ['engine failure after takeoff with emergency return',
  'engine failure after takeoff with emergency return aviation incident report',
  'engine failure after takeoff with emergency return event narrative',
  'engine failure after takeoff with emergency return safety occurrence summary'],
 'rewritten_query': 'aviation incident narrative lookup for: engine failure after takeoff with emergency return',
 'retrieval_plan': {'strategy': 'semantic',
  'fallback_strategy': 'hybrid',
  'top_k': 10,
  'filters': {},
  'routing_reason': 'Narrative incident queries benefit from semantic similarity over safety reports.'},
 'created_at': '2026-06-06T06:14:28.144450'}

## Mock Data (chỉ khi retrieval không chạy được)

**Mock library:** `aviation_rag/phase2_san_contract_adapter.py` → `MOCK_RETRIEVAL_LIBRARY`

Notebook và Streamlit đều gọi `resolve_phase2_output()`:

1. Tìm row `query_id` trong `artifacts/phase2_san_retrieval_output.jsonl`
2. Nếu không có → thử **local FAISS retrieval trên ASRS** (giống Streamlit)
3. Chỉ khi index lỗi → giữ mock + `adapter_mode=generated_mock`

Cell dưới liệt kê mock library để tham khảo contract, không phải nguồn chính khi retrieval available.


In [3]:
mock_inventory = []
for intent, docs in MOCK_RETRIEVAL_LIBRARY.items():
    for doc in docs:
        mock_inventory.append({
            'intent': intent,
            'doc_id': doc['doc_id'],
            'chunk_id': doc['chunk_id'],
            'document_type': doc.get('metadata', {}).get('document_type'),
            'source': doc.get('metadata', {}).get('source'),
            'final_score': doc.get('scores', {}).get('final'),
        })

contract_shape = {
    'MiddleAgentOutput': ['query_id', 'predicted_intent', 'topk_docs', 'retrieval_diagnostics', 'created_at'],
    'RetrievedDoc': ['doc_id', 'chunk_id', 'chunk_text', 'scores', 'metadata'],
    'Required diagnostics for mock': ['adapter_mode=generated_mock', 'contract_owner', 'strategy_requested', 'fallback_strategy'],
}

print(contract_shape)
pd.DataFrame(mock_inventory)


{'MiddleAgentOutput': ['query_id', 'predicted_intent', 'topk_docs', 'retrieval_diagnostics', 'created_at'], 'RetrievedDoc': ['doc_id', 'chunk_id', 'chunk_text', 'scores', 'metadata'], 'Required diagnostics for mock': ['adapter_mode=generated_mock', 'contract_owner', 'strategy_requested', 'fallback_strategy']}


,intent,doc_id,chunk_id,document_type,source,final_score
0,Incident_Report,mock_incident_001,mock_incident_001#0,incident_report,phase2_mock,0.88
1,Incident_Report,mock_incident_002,mock_incident_002#0,incident_report,phase2_mock,0.82
2,Technical_Procedure,mock_procedure_001,mock_procedure_001#0,procedure,phase2_mock,0.91
3,Technical_Procedure,mock_procedure_002,mock_procedure_002#0,procedure,phase2_mock,0.84
4,Metadata_Query,mock_metadata_001,mock_metadata_001#0,metadata,phase2_mock,0.78
5,Metadata_Query,mock_metadata_002,mock_metadata_002#0,metadata,phase2_mock,0.74
6,Factoid,mock_factoid_001,mock_factoid_001#0,factoid,phase2_mock,0.83
7,Factoid,mock_factoid_002,mock_factoid_002#0,factoid,phase2_mock,0.78


## Resolve Phase 2 (đồng bộ Streamlit)

```python
phase2_output = resolve_phase2_output(settings, phase1_output, ...)
```

Kỳ vọng: `adapter_mode=faiss_retrieval`, `first_doc_source` là ASRS doc id (không phải `phase2_mock`).


In [4]:
phase2_rows = []
for row in phase1_rows:
    phase1_output = InputAgentOutput.model_validate(row)
    phase2_rows.append(
        resolve_phase2_output(
            settings,
            phase1_output,
            phase2_retrieval=phase2_retrieval,
            phase2_adapter=phase2_adapter,
        )
    )

write_jsonl(settings.phase2_output_path, phase2_rows)
pd.DataFrame([
    {
        'query_id': row.query_id,
        'predicted_intent': row.predicted_intent,
        'adapter_mode': row.retrieval_diagnostics.get('adapter_mode', 'artifact_or_local'),
        'embedding_backend': row.retrieval_diagnostics.get('embedding_backend'),
        'topk_docs': len(row.topk_docs),
        'first_doc_id': row.topk_docs[0].doc_id if row.topk_docs else None,
        'first_doc_source': row.topk_docs[0].metadata.get('source') if row.topk_docs else None,
        'first_doc_type': row.topk_docs[0].metadata.get('document_type') if row.topk_docs else None,
    }
    for row in phase2_rows
])


,query_id,predicted_intent,adapter_mode,embedding_backend,topk_docs,first_doc_id,first_doc_source,first_doc_type
0,q_incident_001,Incident_Report,faiss_retrieval,tfidf_svd_faiss_fallback,10,667211,asrs_dataset,incident_report
1,q_procedure_001,Technical_Procedure,faiss_retrieval,tfidf_svd_faiss_fallback,10,695230,asrs_dataset,procedure
2,q_metadata_001,Metadata_Query,faiss_retrieval,tfidf_svd_faiss_fallback,10,698971,asrs_dataset,metadata
3,q_factoid_001,Factoid,faiss_retrieval,tfidf_svd_faiss_fallback,10,675768,asrs_dataset,procedure
4,q_e3ee46aa,Incident_Report,faiss_retrieval,tfidf_svd_faiss_fallback,5,695580,asrs_dataset,incident_report
5,q_9508ac11,Incident_Report,faiss_retrieval,tfidf_svd_faiss_fallback,5,695580,asrs_dataset,incident_report
6,q_12d4fe7d,Incident_Report,faiss_retrieval,tfidf_svd_faiss_fallback,5,695580,asrs_dataset,incident_report
7,q_2ea4f089,Incident_Report,faiss_retrieval,tfidf_svd_faiss_fallback,5,695580,asrs_dataset,incident_report


## Generate Phase 3 Grounded Output

`force_local = (RUN_MODE == 'Fast local')` — **cùng quy tắc Streamlit**:

- **Fast local** → local grounded fallback (không cần API key)
- **Full dense/Route LLM** → gọi Route LLM/OpenRouter nếu có key


In [5]:
phase3 = Phase3HoangGroundedQA(settings)
phase1_lookup = {row['query_id']: row for row in phase1_rows}
force_local = force_local_for_run_mode(RUN_MODE)
phase3_rows = []
for phase2_output in phase2_rows:
    query_raw = phase1_lookup[phase2_output.query_id]['query_raw']
    phase3_rows.append(
        phase3.generate(
            question=query_raw,
            middle_output=phase2_output,
            allow_fallback=True,
            force_local=force_local,
        )
    )

pd.DataFrame([
    {
        'query_id': row.query_id,
        'run_mode': RUN_MODE,
        'force_local': force_local,
        'answer_preview': row.answer[:160],
        'citation_count': len(row.citations),
        'hallucination_risk': round(float(row.hallucination_risk), 4),
        'route_llm': (row.grounding_report or {}).get('route_llm_model') or 'local fallback',
    }
    for row in phase3_rows
])


,query_id,run_mode,force_local,answer_preview,citation_count,hallucination_risk,route_llm
0,q_incident_001,Fast local,True,Local grounded fallback: Route LLM is not conf...,3,0.2403,local fallback
1,q_procedure_001,Fast local,True,Local grounded fallback: Route LLM is not conf...,3,0.2326,local fallback
2,q_metadata_001,Fast local,True,Local grounded fallback: Route LLM is not conf...,3,0.1901,local fallback
3,q_factoid_001,Fast local,True,Local grounded fallback: Route LLM is not conf...,3,0.2101,local fallback
4,q_e3ee46aa,Fast local,True,Local grounded fallback: Route LLM is not conf...,3,0.2047,local fallback
5,q_9508ac11,Fast local,True,Local grounded fallback: Route LLM is not conf...,3,0.2047,local fallback
6,q_12d4fe7d,Fast local,True,Local grounded fallback: Route LLM is not conf...,3,0.2047,local fallback
7,q_2ea4f089,Fast local,True,Local grounded fallback: Route LLM is not conf...,3,0.2047,local fallback


## Export Phase 3 Artifact

Output chuẩn: `FinalOutput` (answer, citations, hallucination_risk, grounding_report).


In [6]:
write_jsonl(settings.phase3_output_path, phase3_rows)
required = ['query_id', 'answer', 'citations', 'hallucination_risk', 'grounding_report', 'created_at']
for row in phase3_rows:
    payload = row.model_dump(mode='json')
    missing = [key for key in required if key not in payload]
    if missing:
        raise ValueError(f'Missing keys in {payload.get("query_id")}: {missing}')
print(f'Wrote {len(phase3_rows)} rows to {settings.phase3_output_path}')
print('Phase 3 contract validation passed.')


Wrote 8 rows to C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase3_hoang_grounded_answer_output.jsonl
Phase 3 contract validation passed.


## Hallucination-Risk Proxy — giải thích đánh giá grounding

Phần này **không đánh giá toàn bộ RAG** (Phase 1 intent, Phase 2 retrieval đúng/sai). Chỉ đo **một khía cạnh của Phase 3**: câu trả lời có *bám từ vựng* vào tài liệu Phase 2 retrieve hay không.

### Kỹ thuật (phương pháp)

**Lexical token-overlap** — bag-of-words set intersection, trong `aviation_rag/phase3_hoang_grounded_qa.py` (`_grounding_metrics`).

Không dùng: LLM-as-judge, NLI, embedding similarity, human annotation từng câu.

### Input → Process → Output

| Bước | Nội dung |
|------|----------|
| **Input 1** | `answer` — câu trả lời Phase 3 (LLM hoặc local fallback) |
| **Input 2** | `context` — ghép `chunk_text` của `topk_docs` từ Phase 2 |
| **Process** | Tokenize regex `[a-z0-9]+`, lowercase → lấy **set** token duy nhất → tính giao 2 tập |
| **Output** | `overlap_ratio` và `hallucination_risk` (số thực 0–1), lưu vào `FinalOutput` |

```text
overlap_ratio = |tokens(answer) ∩ tokens(context)| / |tokens(answer)|
hallucination_risk = 1 - overlap_ratio
```

### Ý nghĩa các cột trong bảng debug (cell dưới)

| Cột | Ý nghĩa |
|-----|---------|
| `answer_token_count` | Số token **khác nhau** trong answer |
| `context_token_count` | Số token khác nhau trong toàn bộ context retrieve |
| `overlap_token_count` | Số token vừa có trong answer vừa có trong context |
| `overlap_ratio` | Tỷ lệ token answer nằm trong context (càng cao càng bám nguồn) |
| `hallucination_risk` | Proxy rủi ro thông tin ngoài context (càng thấp càng tốt) |

### Ví dụ đọc số: `q_incident_001` → `0.2403`

- Answer: 129 token · Context: 494 token · Trùng: 98 token
- `overlap_ratio = 98/129 ≈ 0.7597` → ~**76%** từ trong answer có trong tài liệu retrieve
- `hallucination_risk = 1 - 0.7597 = **0.2403**` → ~**24%** token answer **không** xuất hiện trong context

**Lưu ý:** đây là heuristic lexical, không phải xác suất hallucination thật. Paraphrase đúng có thể bị tính risk cao; copy từ context sai ngữ cảnh vẫn có thể risk thấp.

Nếu nhiều `query_id` có cùng metric (vd. 4 dòng hash cuối), thường do **cùng answer template** hoặc **cùng context retrieve** — kiểm tra `phase3_rows` / `phase2_rows` tương ứng.


In [7]:
def token_set(text: str) -> set[str]:
    return set(re.findall(r'[a-z0-9]+', (text or '').lower()))

risk_debug_rows = []
for phase3_row, phase2_output in zip(phase3_rows, phase2_rows):
    answer_tokens = token_set(phase3_row.answer)
    context_text = ' '.join(doc.chunk_text for doc in phase2_output.topk_docs)
    context_tokens = token_set(context_text)
    overlap = answer_tokens.intersection(context_tokens)
    risk_debug_rows.append({
        'query_id': phase3_row.query_id,
        'answer_token_count': len(answer_tokens),
        'context_token_count': len(context_tokens),
        'overlap_token_count': len(overlap),
        'overlap_ratio': round(len(overlap) / max(1, len(answer_tokens)), 4),
        'hallucination_risk': round(float(phase3_row.hallucination_risk), 4),
    })

pd.DataFrame(risk_debug_rows)


,query_id,answer_token_count,context_token_count,overlap_token_count,overlap_ratio,hallucination_risk
0,q_incident_001,129,494,98,0.7597,0.2403
1,q_procedure_001,129,642,99,0.7674,0.2326
2,q_metadata_001,142,1030,115,0.8099,0.1901
3,q_factoid_001,119,625,94,0.7899,0.2101
4,q_e3ee46aa,127,391,101,0.7953,0.2047
5,q_9508ac11,127,391,101,0.7953,0.2047
6,q_12d4fe7d,127,391,101,0.7953,0.2047
7,q_2ea4f089,127,391,101,0.7953,0.2047


## Đánh giá Phase 3 — tiêu chí tổng hợp (descriptive metrics)

Cell dưới tính **thống kê mô tả** trên toàn bộ sample Phase 3 (không phải pass/fail từng câu).

### Tiêu chí và cách đo

| Tiêu chí | Cách đo | Ý nghĩa |
|----------|---------|---------|
| **Citation coverage** | % query có ≥ 1 citation | Output có trích nguồn không |
| **Avg citation count** | Trung bình số citation | Mức độ trích dẫn |
| **Avg hallucination risk** | Trung bình `hallucination_risk` (lexical overlap proxy ở section trên) | Mức bám context trung bình của batch |
| **Empty-answer rate** | % answer rỗng | Pipeline có trả lời được không |

### Tiêu chí **không** nằm ở đây

- Câu trả lời đúng/sai về mặt kiến thức hàng không
- Retrieval có lấy đúng document không (Phase 2)
- Intent routing (Phase 1)
- Citation có trích đúng đoạn liên quan không

Pass/fail theo gold set → section **Gold-set Grounding Evaluation** bên dưới.


In [8]:
def as_dict(row):
    if hasattr(row, 'model_dump'):
        return row.model_dump()
    if isinstance(row, dict):
        return row
    raise TypeError(f'Unsupported phase3 row type: {type(row)!r}')

quality_rows = []
for raw_row in phase3_rows:
    row = as_dict(raw_row)
    answer = str(row.get('answer', ''))
    citations = row.get('citations', []) or []
    risk = float(row.get('hallucination_risk', 0.0) or 0.0)
    quality_rows.append({
        'query_id': row.get('query_id'),
        'answer_chars': len(answer),
        'citation_count': len(citations),
        'has_citation': bool(citations),
        'hallucination_risk': round(risk, 4),
        'empty_answer': len(answer.strip()) == 0,
    })

quality_frame = pd.DataFrame(quality_rows)
summary = {
    'sample_size': len(quality_rows),
    'citation_coverage': round(quality_frame['has_citation'].mean(), 4),
    'avg_citation_count': round(quality_frame['citation_count'].mean(), 4),
    'avg_hallucination_risk': round(quality_frame['hallucination_risk'].mean(), 4),
    'empty_answer_rate': round(quality_frame['empty_answer'].mean(), 4),
}
print(summary)
quality_frame


{'sample_size': 8, 'citation_coverage': np.float64(1.0), 'avg_citation_count': np.float64(3.0), 'avg_hallucination_risk': np.float64(0.2115), 'empty_answer_rate': np.float64(0.0)}


,query_id,answer_chars,citation_count,has_citation,hallucination_risk,empty_answer
0,q_incident_001,1157,3,True,0.2403,False
1,q_procedure_001,1145,3,True,0.2326,False
2,q_metadata_001,1162,3,True,0.1901,False
3,q_factoid_001,1146,3,True,0.2101,False
4,q_e3ee46aa,1156,3,True,0.2047,False
5,q_9508ac11,1156,3,True,0.2047,False
6,q_12d4fe7d,1156,3,True,0.2047,False
7,q_2ea4f089,1156,3,True,0.2047,False


## Gold-set Grounding Evaluation — tiêu chí pass/fail

Đánh giá **rule-based** trên `data/phase3_grounding_gold_labels.jsonl` qua `aviation_rag/phase3_grounding_eval.py`.

Mỗi query gold có **3 tiêu chí bắt buộc**; `passed = True` chỉ khi cả 3 đúng:

| Tiêu chí | Field gold | Ngưỡng mặc định |
|----------|------------|-----------------|
| Answer không rỗng | `require_non_empty_answer` | phải có text |
| Đủ citation | `min_citations` | ≥ 1 citation |
| Risk trong ngưỡng | `max_hallucination_risk` | `hallucination_risk` ≤ 0.95 |

`hallucination_risk` ở đây vẫn là **lexical overlap proxy** (section Hallucination-Risk Proxy), không phải human judge.

**Ví dụ:** `q_incident_001` có risk `0.2403` → thấp hơn `0.95` → **đạt** tiêu chí risk.

**Output:** `pass_rate` = tỷ lệ query gold pass cả 3 check. Đây là **contract / sanity check**, không thay thế đánh giá chất lượng nội dung answer.


In [9]:
from aviation_rag.phase3_grounding_eval import evaluate_grounding_gold

grounding_rows = []
for phase3_row in phase3_rows:
    payload = phase3_row.model_dump()
    payload['query_raw'] = phase1_lookup[phase3_row.query_id]['query_raw']
    grounding_rows.append(payload)

grounding_gold = evaluate_grounding_gold(grounding_rows, settings.phase3_gold_labels_path)
print({
    'gold_rows': grounding_gold.get('gold_rows'),
    'pass_rate': grounding_gold.get('pass_rate'),
    'gold_path': grounding_gold.get('gold_path'),
})
pd.DataFrame(grounding_gold.get('gold_eval_rows', []))


{'gold_rows': 4, 'pass_rate': 1.0, 'gold_path': 'C:\\Users\\DELL\\Desktop\\Vinh Hoang\\Master Program\\Xử lý ngôn ngữ tự nhiên\\Project\\data\\phase3_grounding_gold_labels.jsonl'}


,query_id,query_raw,citation_count,hallucination_risk,checks,passed
0,q_incident_001,engine failure after takeoff with emergency re...,3,0.2403,"{'non_empty_answer': True, 'min_citations': Tr...",True
1,q_procedure_001,den bao ENG OIL PRESS sang thi lam gi?,3,0.2326,"{'non_empty_answer': True, 'min_citations': Tr...",True
2,q_metadata_001,crosswind turbulence during final approach at ...,3,0.1901,"{'non_empty_answer': True, 'min_citations': Tr...",True
3,q_factoid_001,what is the meaning of MEL in aviation?,3,0.2101,"{'non_empty_answer': True, 'min_citations': Tr...",True
